# Brazilian personal data redaction: sample notebook

This notebook shows the model package end to end: a real-time endpoint and batch transform jobs over JSON Lines, CSV and plain-text files. It runs in a SageMaker notebook or SageMaker Studio, uses only sample data written into the notebook, and deletes everything it creates.

**Before you start**

1. Subscribe to the product in AWS Marketplace.
2. Use an IAM role that can run SageMaker endpoints and batch transform jobs and read and write the SageMaker default S3 bucket (`AmazonSageMakerFullAccess` is enough).

**Cost:** about 15 minutes of `ml.m5.large` instances, under $0.10 of AWS charges, plus the software price per instance-hour (none during the free trial).

**What it finds:** CPF, CNPJ (including the alphanumeric format), RG, CNH, PIS/PASEP, voter ID, OAB, RENAVAM, vehicle plates, bank branch and account, card numbers, phone numbers, emails, CEP, addresses, PIX keys, dates of birth and people's names, in Portuguese text. The model runs with no network access; nothing leaves your account.

In [ ]:
import json
import time

import boto3
from botocore.exceptions import ClientError

session = boto3.Session()
region = session.region_name
account = session.client("sts").get_caller_identity()["Account"]
sm = session.client("sagemaker")
runtime = session.client("sagemaker-runtime")
s3 = session.client("s3")


def execution_role():
    """The IAM role of this SageMaker notebook instance or Studio space."""
    try:
        with open("/opt/ml/metadata/resource-metadata.json") as f:
            meta = json.load(f)
    except FileNotFoundError:
        raise SystemExit("Run this notebook in SageMaker, or set role to an IAM role ARN that SageMaker can use.")
    if meta.get("ExecutionRoleArn"):
        return meta["ExecutionRoleArn"]
    if meta.get("DomainId"):
        if meta.get("UserProfileName"):
            user = sm.describe_user_profile(DomainId=meta["DomainId"], UserProfileName=meta["UserProfileName"])
            if user.get("UserSettings", {}).get("ExecutionRole"):
                return user["UserSettings"]["ExecutionRole"]
        return sm.describe_domain(DomainId=meta["DomainId"])["DefaultUserSettings"]["ExecutionRole"]
    return sm.describe_notebook_instance(NotebookInstanceName=meta["ResourceName"])["RoleArn"]


role = execution_role()

# SageMaker's default bucket for this account and Region, created if missing.
bucket = f"sagemaker-{region}-{account}"
try:
    s3.head_bucket(Bucket=bucket)
except ClientError:
    s3.create_bucket(Bucket=bucket, **({} if region == "us-east-1" else
                                       {"CreateBucketConfiguration": {"LocationConstraint": region}}))
prefix = "brazilian-pii-redaction-sample"

# The model package ARN for each Region, from the product's Marketplace listing.
MODEL_PACKAGE_ARNS = {
    "us-east-1": "arn:aws:sagemaker:us-east-1:865070037744:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "us-east-2": "arn:aws:sagemaker:us-east-2:057799348421:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "us-west-1": "arn:aws:sagemaker:us-west-1:382657785993:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "us-west-2": "arn:aws:sagemaker:us-west-2:594846645681:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "ca-central-1": "arn:aws:sagemaker:ca-central-1:470592106596:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "sa-east-1": "arn:aws:sagemaker:sa-east-1:270155090741:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "eu-central-1": "arn:aws:sagemaker:eu-central-1:446921602837:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "eu-west-1": "arn:aws:sagemaker:eu-west-1:985815980388:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "eu-west-2": "arn:aws:sagemaker:eu-west-2:856760150666:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "eu-west-3": "arn:aws:sagemaker:eu-west-3:843114510376:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "eu-north-1": "arn:aws:sagemaker:eu-north-1:136758871317:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "ap-south-1": "arn:aws:sagemaker:ap-south-1:077584701553:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "ap-southeast-1": "arn:aws:sagemaker:ap-southeast-1:192199979996:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "ap-southeast-2": "arn:aws:sagemaker:ap-southeast-2:666831318237:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "ap-northeast-1": "arn:aws:sagemaker:ap-northeast-1:977537786026:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
    "ap-northeast-2": "arn:aws:sagemaker:ap-northeast-2:745090734665:model-package/brazilian-pii-redaction-1-0-1-5ae2d7809a6d31fcb4870592c5eae556",
}
model_package_arn = MODEL_PACKAGE_ARNS[region]
print(region, role, model_package_arn)

## 1. Sample data

Fake Brazilian personal data in typical messages: support chats, forms, payments, a vehicle record. The last sentence has an order protocol number that must be left alone.

In [ ]:
samples = [
 "Olá, sou a Fernanda Ribeiro, CPF 937.775.749-53, e quero cancelar a assinatura do plano anual.",
 "Cliente Marcos Vinícius Teixeira ligou do 21972016205 pedindo a segunda via do boleto.",
 "Enviar o contrato para o e-mail julianacosta@example.com até sexta-feira.",
 "Fornecedor: Gráfica Horizonte Ltda., CNPJ 65.764.346/0001-01.",
 "meu cpf é 19044421298 e o pedido 48213377 ainda não chegou",
 "Paciente nascida em 14/03/1987, RG 47.376.695-4 SSP/SP, atendida na unidade Centro.",
 "Reembolso via PIX para a chave bd98b6c5-d384-4b5a-8f69-546280d6f34c, titular Ana Paula Moreira.",
 "Depósito na agência 9382, conta corrente 38731-X, em nome de Ricardo Alves.",
 "Veículo placa TNP5X02, RENAVAM 45875283810, multado em 02/09/2026.",
 "Endereço de entrega: Rua das Acácias, 245, apto 31, Jardim Botânico, Curitiba/PR, CEP 80531-615.",
 "Cartão final 8980 recusado; o cliente informou o cartão 5588 9902 6033 2177.",
 "O protocolo 2026114455 foi aberto e o prazo de resposta é de 5 dias úteis."
]
for s in samples:
    print(s)

## 2. Create the model

The model runs with network isolation, as every Marketplace model does. `key` is your secret for placeholders: the same value always gets the same placeholder (for example `[CPF_3f2a1c]`) with the same key, so records can still be joined after cleaning. Keep it secret, and keep it the same across runs.

AWS Marketplace models take no environment variables, so the key and the options travel with your data: in the JSON body, in each JSON Lines record, or as Content-Type parameters such as `text/csv; key=...; csv_columns=2`. Without a key, the model makes a temporary one at start-up and says so in every response.

In [ ]:
key = "replace-with-your-own-secret"
name = f"brazilian-pii-redaction-{int(time.time())}"
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=role,
    EnableNetworkIsolation=True,
    PrimaryContainer={"ModelPackageName": model_package_arn},
)
print(name)

## 3. Real-time endpoint

Creating the endpoint takes about 5 to 8 minutes.

In [ ]:
sm.create_endpoint_config(EndpointConfigName=name, ProductionVariants=[{
    "VariantName": "main", "ModelName": name, "InstanceType": "ml.m5.large", "InitialInstanceCount": 1}])
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
sm.get_waiter("endpoint_in_service").wait(EndpointName=name)
print("endpoint in service")

Send JSON: `{"text": "..."}` for one text, or `{"texts": [...]}` for several, with your `key` and optional `options`. By default every finding is replaced with a placeholder and the original values are not returned.

In [ ]:
def redact(texts, options=None):
    body = {"texts": texts, "key": key, **({"options": options} if options else {})}
    r = runtime.invoke_endpoint(EndpointName=name, ContentType="application/json", Body=json.dumps(body).encode())
    return json.loads(r["Body"].read())

result = redact(samples)
for r in result["results"]:
    print(r["text"])

Each result also lists what was found, with its position in the original text:

In [ ]:
print(json.dumps(result["results"][0]["findings"], indent=1))

### Options

- `action`: `replace` (default), `mask` (numbers keep their last two characters, CPF follows the government format `***.456.789-**`, and names, dates of birth, emails and addresses are hidden in full), `remove`, or `report` (leave the text as it is and only list the findings).
- `per_label`: a different action per type, for example keep names but mask CPFs.
- `labels`: only look for these types.
- `include_values`: also return the original values (off by default).

In [ ]:
masked = redact(samples[:2], {"action": "mask"})
for r in masked["results"]:
    print(r["text"])

mixed = redact(samples[7:9], {"per_label": {"PERSON": "report", "BANK_ACCOUNT": "mask"}})
for r in mixed["results"]:
    print(r["text"])

report = redact(samples[:1], {"action": "report", "include_values": True})
print(json.dumps(report["results"][0]["findings"], ensure_ascii=False, indent=1))

Plain text works too: send `text/plain` with the key as a Content-Type parameter and, with `Accept: text/plain`, get only the cleaned text back. Options can go there as well, for example `text/plain; key=...; action=mask`.

In [ ]:
r = runtime.invoke_endpoint(EndpointName=name, ContentType=f"text/plain; key={key}", Accept="text/plain",
                            Body=samples[5].encode("utf-8"))
print(r["Body"].read().decode("utf-8"))

Delete the endpoint when done; you pay for it while it runs.

In [ ]:
sm.delete_endpoint(EndpointName=name)
sm.delete_endpoint_config(EndpointConfigName=name)

## 4. Batch transform

Batch transform reads files from S3, cleans them and writes the results to S3, then stops. The same samples are written as three files:

- JSON Lines: one `{"id": ..., "text": ..., "key": ...}` per line, with optional `"options"`, returned as one line each with the cleaned text and the findings.
- CSV: every field is cleaned unless `csv_columns` says which (counting from 0); the output has the same columns. The key and options go in the Content-Type: `text/csv; key=...; csv_columns=2`.
- Plain text: one text per line, returned as cleaned lines, with `text/plain; key=...`.

Content-Type parameters work for JSON Lines too, and cover every line that carries no key or options of its own. Lists are written with commas (`labels=CPF,PHONE`), and an action per type as `per_label=PERSON:report,CPF:mask`. Set `Accept` to the plain type. Use `SplitType="Line"`. CSV fields with line breaks inside quotes need `SplitType="None"` instead. The three jobs run in parallel and take about 5 to 7 minutes.

In [ ]:
import csv
import io

buf = io.StringIO()
writer = csv.writer(buf, lineterminator="\n")
writer.writerow(["id", "channel", "message"])
writer.writerows([f"msg-{i:03d}", "chat", s] for i, s in enumerate(samples, 1))
files = {
    "jsonl": ("application/jsonlines", "".join(json.dumps({"id": f"msg-{i:03d}", "text": s, "key": key},
                                                          ensure_ascii=False) + "\n" for i, s in enumerate(samples, 1))),
    "csv": (f"text/csv; key={key}; csv_columns=2", buf.getvalue()),
    "txt": (f"text/plain; key={key}", "\n".join(samples) + "\n"),
}
jobs = {}
for kind, (content_type, body) in files.items():
    s3_key = f"{prefix}/input/{kind}/sample.{kind}"
    s3.put_object(Bucket=bucket, Key=s3_key, Body=body.encode("utf-8"))
    job = f"{name}-{kind}-{int(time.time())}"[:63]  # job names must be unique, so a rerun gets new ones
    sm.create_transform_job(
        TransformJobName=job, ModelName=name,
        TransformInput={"DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{bucket}/{s3_key}"}},
                        "ContentType": content_type, "SplitType": "Line"},
        TransformOutput={"S3OutputPath": f"s3://{bucket}/{prefix}/output/{kind}/", "Accept": content_type.split(";")[0],
                         "AssembleWith": "Line"},
        TransformResources={"InstanceType": "ml.m5.large", "InstanceCount": 1})
    jobs[kind] = job
for kind, job in jobs.items():
    sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=job)
    print(kind, sm.describe_transform_job(TransformJobName=job)["TransformJobStatus"])

In [ ]:
for kind in jobs:
    s3_key = f"{prefix}/output/{kind}/sample.{kind}.out"
    print(f"--- {kind}")
    print(s3.get_object(Bucket=bucket, Key=s3_key)["Body"].read().decode("utf-8")[:1500])

## 5. Clean up

In [ ]:
sm.delete_model(ModelName=name)
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket=bucket, Key=obj["Key"])
print("deleted the model and the sample files")